In [18]:
import torch 
import torch.nn as nn 
from medmnist import PathMNIST
import torchvision.transforms as  transforms 
from torch.utils.data import DataLoader 
import torchvision.models as models 



In [19]:
data_transforms = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean = [0.5,0.5,0.5], std = [0.5,0.5,0.5])]) 


In [20]:
Train_dataset = PathMNIST(split = 'train', as_rgb = True ,download= True ,transform= data_transforms)
Val_dataset = PathMNIST ( split = 'val', as_rgb=True , download= True ,transform= data_transforms)


In [21]:
Train_loader = DataLoader (Train_dataset, batch_size= 64 ,shuffle = True )
Val_loader = DataLoader ( Val_dataset, batch_size = 64 , shuffle = True )


In [27]:
class ResNet18(nn.Module)  : 
    def __init__ ( self, num_classes=9 , pretrained = True ) : 
        super(ResNet18,self ).__init__()
        self.resnet =  models.resnet18(pretrained = pretrained )
        self.resnet.conv1 = nn.Conv2d(3,64,kernel_size =3 , stride =1 , padding =1 , bias = False )
        self.resnet.maxpool = nn.Identity() 
        in_feature = self.resnet.fc.in_features 
        self.resnet.fc =  nn.Linear(in_feature, num_classes )
    def forward (self , x ) : 
        return self.resnet(x) 

In [28]:
model = ResNet18(num_classes=9)



/home/dung/miniconda3/envs/tqf/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dung/miniconda3/envs/tqf/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [29]:
dummy_input = torch.randn(16, 3, 28, 28) # Batch=16
output = model(dummy_input)
print("Kích thước đầu ra:", output.shape) # Kỳ vọng: [16, 9]

Kích thước đầu ra: torch.Size([16, 9])


In [ ]:
import torch
from tqdm import tqdm # Thư viện tạo thanh tiến trình cực đẹp

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 1. Chuyển mô hình sang chế độ huấn luyện
    # Rất quan trọng nếu mô hình có Dropout hoặc BatchNorm
    model.train() 
    
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Bọc dataloader bằng tqdm để hiển thị tiến trình
    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for inputs, targets in pbar:
        # 2. Đẩy dữ liệu lên thiết bị (GPU hoặc CPU)
        inputs = inputs.to(device)
        targets = targets.to(device)
        

        targets = targets.squeeze().long()
        
        # 3. Xóa các gradient từ bước (batch) trước
        optimizer.zero_grad()
        
        # 4. Forward pass: Tính toán đầu ra của mô hình
        outputs = model(inputs)
        
        # 5. Tính hàm mất mát (Loss)
        loss = criterion(outputs, targets)
        
        # 6. Backward pass: Lan truyền ngược để tính Gradient
        loss.backward()
        
        # 7. Cập nhật trọng số của mô hình
        optimizer.step()
        
        # --- Phần tính toán thống kê (Metric) ---
        # Cộng dồn loss (nhân với kích thước batch để tính trung bình cho chính xác ở cuối)
        running_loss += loss.item() * inputs.size(0)
        
        # Tính số lượng dự đoán đúng
        _, predicted = torch.max(outputs.data, 1)
        total_samples += targets.size(0)
        correct_preds += (predicted == targets).sum().item()
        
        # Cập nhật thanh tiến trình liên tục
        pbar.set_postfix({'loss': loss.item()})
        
    # Tính Loss và Accuracy trung bình của toàn bộ Epoch
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    
    return epoch_loss, epoch_acc

In [31]:
import torch

def validate_one_epoch(model, dataloader, criterion, device):
    # 1. Chuyển sang chế độ đánh giá (Tắt Dropout, cố định BatchNorm)
    model.eval()
    
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # 2. TẮT tính toán Gradient (Cực kỳ quan trọng để không bị tràn RAM/VRAM)
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.squeeze().long().to(device) # Xử lý nhãn MedMNIST
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Thống kê
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += targets.size(0)
            correct_preds += (predicted == targets).sum().item()
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    
    return epoch_loss, epoch_acc

In [32]:
import copy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20):
    best_acc = 0.0
    # Lưu lại bộ trọng số tốt nhất
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # 1. Train
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        
        # 2. Validate
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
        
        # 3. Cập nhật Learning Rate (nếu dùng Scheduler)
        if scheduler is not None:
            scheduler.step()
            
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}')
        
        # 4. Lưu model tốt nhất dựa trên Validation Accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            # Xuất file ra ổ cứng
            torch.save(model.state_dict(), 'best_pathmnist_model.pth')
            print(">>> Đã lưu model mới tốt nhất!")

    print(f'\nQuá trình huấn luyện hoàn tất! Độ chính xác Val cao nhất: {best_acc:.4f}')
    
    # Load lại trọng số tốt nhất vào mô hình trước khi trả về
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
import torch.optim as optim

# 1. Cấu hình thiết bị (Ưu tiên dùng GPU nếu cóimport copy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20):
    best_acc = 0.0
    # Lưu lại bộ trọng số tốt nhất
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # 1. Train
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        
        # 2. Validate
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
        
        # 3. Cập nhật Learning Rate (nếu dùng Scheduler)
        if scheduler is not None:
            scheduler.step()
            
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}')
        
        # 4. Lưu model tốt nhất dựa trên Validation Accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            # Xuất file ra ổ cứng
            torch.save(model.state_dict(), 'best_pathmnist_model.pth')
            print(">>> Đã lưu model mới tốt nhất!")

    print(f'\nQuá trình huấn luyện hoàn tất! Độ chính xác Val cao nhất: {best_acc:.4f}')
    
    # Load lại trọng số tốt nhất vào mô hình trước khi trả về
    model.load_state_dict(best_model_wts)
    return model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

# Đưa mô hình lên thiết bị (Giả sử bạn đã khởi tạo biến 'model' là mạng ResNet+LSTM)
model = model.to(device)

# 2. Số lượng Epoch
# Với PathMNIST 28x28, 20-30 epochs thường là đủ để hội tụ
NUM_EPOCHS = 30 

# 3. Hàm mất mát (Loss Function)
# PathMNIST có một chút mất cân bằng lớp, ta có thể dùng label_smoothing để mô hình bớt overconfident
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 4. Bộ tối ưu hóa (Optimizer)
# Khuyến nghị dùng AdamW thay vì Adam thường để mô hình tổng quát hóa tốt hơn
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 5. Bộ điều chỉnh Learning Rate (Scheduler) - Vũ khí bí mật!
# CosineAnnealingLR sẽ giảm dần LR theo hình sin, giúp mô hình chui lọt vào điểm hội tụ tối ưu nhất
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ==========================================
# BẮT ĐẦU CHẠY HUẤN LUYỆN
# (Giả sử bạn đã có train_loader và val_loader)
# ==========================================
best_model = train_model(
    model=model, 
    train_loader=Train_loader, 
    val_loader=Val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    scheduler=scheduler, 
    device=device, 
    num_epochs=NUM_EPOCHS
)

Đang sử dụng thiết bị: cuda

Epoch 1/30
----------


Training:   0%|          | 0/1407 [00:00<?, ?it/s]

Train Loss: 0.7236 | Train Acc: 0.8999
Val Loss:   0.5927 | Val Acc:   0.9559
>>> Đã lưu model mới tốt nhất!

Epoch 2/30
----------


Train Loss: 0.5996 | Train Acc: 0.9537
Val Loss:   0.5653 | Val Acc:   0.9680
>>> Đã lưu model mới tốt nhất!

Epoch 3/30
----------


Train Loss: 0.5676 | Train Acc: 0.9671
Val Loss:   0.5618 | Val Acc:   0.9685
>>> Đã lưu model mới tốt nhất!

Epoch 4/30
----------


Train Loss: 0.5497 | Train Acc: 0.9748
Val Loss:   0.5572 | Val Acc:   0.9702
>>> Đã lưu model mới tốt nhất!

Epoch 5/30
----------


Train Loss: 0.5345 | Train Acc: 0.9805
Val Loss:   0.5811 | Val Acc:   0.9589

Epoch 6/30
----------


Train Loss: 0.5243 | Train Acc: 0.9847
Val Loss:   0.5514 | Val Acc:   0.9708
>>> Đã lưu model mới tốt nhất!

Epoch 7/30
----------


KeyboardInterrupt: 